## Introduction to APIs

As a crypto accountant, much of your work starts with **getting data** — deposits and withdrawals from exchanges, on-chain transactions from blockchain explorers, token prices for valuation, and so on.

Today, that often means:

- Logging into an exchange and clicking **Export CSV**
- Asking a client to export data for you
- Copy-pasting from a block explorer one transaction at a time

These approaches work, but they are **slow**, **manual**, and **hard to repeat** when you need fresh data every month.

An **API** (Application Programming Interface) is the programmatic way to fetch that same data — directly from the source, on demand, without using the website's buttons.

In this notebook we'll cover:

1. What an API is (in plain language)
2. How a web request works
3. Making your first API call in Python
4. Turning API data into a pandas DataFrame (connecting to Session 1)
5. Practical considerations: API keys, pagination, and rate limits

---

**Prerequisites:** You should be comfortable with Jupyter notebooks and basic pandas from `01_intro_to_jupyter_and_pandas`.

## What is an API?

Think of a **website GUI** as dining in a restaurant. You browse the **menu** (options of available data via **endpoints**) and tell the **waiter** what you want. The waiter then carries this message behind the scenes into the kitchen, and returns to you with the requested product (data).

An **API** is that **waiter** — the middle layer that translates between what you ask for and what the system (the kitchen) delivers.

| Approach | How you get data | Good for |
|----------|------------------|----------|
| **GUI export** | Click buttons, download a file | One-off exports, small datasets |
| **API** | Send a structured request, receive structured data | Automation, large datasets, recurring jobs |

The data you receive is usually in **JSON** format — a structured text format that Python (and pandas) can read easily. It looks similar to a nested dictionary. (Don't worry, we'll look at some examples soon).

## How a web request works

Most APIs you'll use are **REST APIs** accessed over the internet using **HTTP** — the same protocol your browser uses when you visit a website.

The key idea is simple:

1. You send a **request** to a **URL** (a web address)
2. The server sends back a **response** (usually JSON data)

For example, to get the current Bitcoin price from CoinGecko, the URL might look like:

```
https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd
```

Breaking that down:

| Part | Meaning |
|------|--------|
| `https://api.coingecko.com` | The server (CoinGecko's API) |
| `/api/v3/simple/price` | The specific **endpoint** — what you're asking for |
| `?ids=bitcoin&vs_currencies=usd` | **Parameters** — filters/options for your request |

The most common request type is **GET** — you're *getting* data, not changing anything. That's what we'll focus on today.

As part of the response, the server returns a **status code**. The one you want is **200** (OK). If the status code is not 200, it means the request was not successful. The code will help you narrow down the reason it failed, allowing you to make necessary changes. Other common codes:

| Code | Meaning |
|------|--------|
| 200 | Success |
| 401 | Unauthorized — usually a missing or invalid API key |
| 403 | Forbidden — you don't have permission |
| 404 | Page not found - endpoint/URL doesn't exist |
| 429 | Too many requests — you've hit a **rate limit** |
| 500 | Server error — problem on their end, try again later |

## Making your first API call

In Python, the `requests` library is the standard way to call APIs. Run the cell below to import it (and pandas, which we'll use later).

In [3]:
import requests
import pandas as pd
from pprint import pprint

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

### Example 1: Fetch a token price (CoinGecko)

CoinGecko offers a free public API — no account or API key required for basic use. This makes it ideal for learning.

We'll ask for the USD price of Bitcoin, Ethereum, and Tether.

In [5]:
# The URL (endpoint + parameters)
base_url = "https://api.coingecko.com/api/v3/"

method = "GET"
endpoint = "simple/price"
request_url = f"{base_url}{endpoint}"

# Parameters are passed as a dictionary — requests turns them into ?ids=bitcoin&...
params = {
    "vs_currencies": "usd",
    "ids": "bitcoin,ethereum,solana",
}

# Send the GET request
response = requests.request(method, request_url, params=params)

# Check the status code
print(f"Status code: {response.status_code}")

print(f"Raw response: {response}")

# Parse the JSON response into a Python dictionary
data = response.json()
pprint(data)

Status code: 200
Raw response: <Response [200]>
{'bitcoin': {'usd': 71505},
 'ethereum': {'usd': 1968.1},
 'solana': {'usd': 79.58}}


The response is a **dictionary** (key–value pairs). Each token ID maps to another dictionary with the price:

```json
{
  "bitcoin": {"usd": 97342.15},
  "ethereum": {"usd": 3421.88},
  "tether": {"usd": 1.0003}
}
```

You can access individual values just like any Python dictionary:

In [6]:
btc_price = data["bitcoin"]["usd"]
print(f"Bitcoin price: ${btc_price:,.2f}")

Bitcoin price: $71,505.00


## From API response to pandas DataFrame

A single price lookup is useful as a one-off fetch, but generally our work will require large datasets in the form of **tables** (e.g. CSV, Excel, pandas DataFrame). CoinGecko's [`/coins/markets`](https://docs.coingecko.com/v3.0.1/reference/coins-markets) endpoint returns a **list** of coins with price, market cap, volume, and more — ideal for building a table.

We'll fetch the first page of results and turn the JSON into a pandas DataFrame.

In [9]:
base_url = "https://api.coingecko.com/api/v3/"

method = "GET"
endpoint = "coins/markets"
request_url = f"{base_url}{endpoint}"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 5,
    "page": 1,
}

response = requests.request(method, request_url, params=params)
print(f"Status code: {response.status_code}")

markets_data = response.json()
print(f"Records on this page: {len(markets_data)}")
pprint(markets_data[0])  # show the first coin

Status code: 200
Records on this page: 5
{'ath': 126080,
 'ath_change_percentage': -43.60317,
 'ath_date': '2025-10-06T18:57:42.558Z',
 'atl': 67.81,
 'atl_change_percentage': 104760.79912,
 'atl_date': '2013-07-06T00:00:00.000Z',
 'circulating_supply': 20037046.0,
 'current_price': 71105,
 'fully_diluted_valuation': 1426504072310,
 'high_24h': 74001,
 'id': 'bitcoin',
 'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
 'last_updated': '2026-06-01T15:48:52.472Z',
 'low_24h': 71035,
 'market_cap': 1426495813931,
 'market_cap_change_24h': -47972035862.76099,
 'market_cap_change_percentage_24h': -3.25352,
 'market_cap_rank': 1,
 'max_supply': 21000000.0,
 'name': 'Bitcoin',
 'price_change_24h': -2448.703224443714,
 'price_change_percentage_24h': -3.32913,
 'roi': None,
 'symbol': 'btc',
 'total_supply': 20037162.0,
 'total_volume': 39742423042}


In [10]:
# The response is a list of dictionaries — one per coin
markets_df = pd.DataFrame(markets_data)
markets_df

,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,low_24h,price_change_24h,price_change_percentage_24h,market_cap_change_24h,market_cap_change_percentage_24h,circulating_supply,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,71105.000000,1426495813931,1,1426504072310,39742423042,74001.000000,71035.000000,-2448.703224,-3.32913,-4.797204e+10,-3.25352,2.003705e+07,2.003716e+07,2.100000e+07,126080.00,-43.60317,2025-10-06T18:57:42.558Z,67.810000,1.047608e+05,2013-07-06T00:00:00.000Z,None,2026-06-01T15:48:52.472Z
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628,1969.050000,237933190535,2,237933190535,16231225879,2016.020000,1960.150000,-39.679879,-1.97537,-4.500536e+09,-1.85640,1.206852e+08,1.206852e+08,NaN,4946.05,-60.18934,2025-08-24T19:21:03.333Z,0.432979,4.546692e+05,2015-10-20T00:00:00.000Z,"{'times': 36.02502200330675, 'currency': 'btc', 'percentage': 3602.502200330675}",2026-06-01T15:48:52.544Z
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images/325/large/Tether.png?1696501661,0.998375,187940149618,3,193398405378,65167177645,0.998889,0.998314,-0.000182,-0.01822,-2.625034e+08,-0.13948,1.882392e+11,1.937062e+11,NaN,1.32,-24.54313,2018-07-24T00:00:00.000Z,0.572521,7.438073e+01,2015-03-02T00:00:00.000Z,None,2026-06-01T15:48:52.296Z
3,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,681.020000,91819994568,4,91819993478,2873104456,725.890000,675.990000,-38.428826,-5.34143,-5.263597e+09,-5.42172,1.347845e+08,1.347845e+08,2.000000e+08,1369.99,-50.29029,2025-10-13T08:41:24.131Z,0.039818,1.710242e+06,2017-10-19T00:00:00.000Z,None,2026-06-01T15:48:52.380Z
4,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images/44/large/xrp-symbol-white-128.png?1696501442,1.280000,79597193489,5,128410044927,1802785294,1.340000,1.280000,-0.045955,-3.45527,-2.934757e+09,-3.55590,6.197784e+10,9.998566e+10,1.000000e+11,3.65,-64.78554,2025-07-18T03:40:53.808Z,0.002686,4.770169e+04,2014-05-22T00:00:00.000Z,None,2026-06-01T15:48:52.228Z


The API returns many columns — far more than you typically need. For accounting and valuation, the useful ones are usually:

| Column | What it is |
|--------|------------|
| `id` | CoinGecko ID (matches the `coingecko_id` column in your transfer data) |
| `symbol` | Ticker (BTC, ETH, …) |
| `name` | Full name |
| `current_price` | Spot price in USD |
| `market_cap` | Market capitalisation |
| `total_volume` | 24h trading volume |
| `price_change_percentage_24h` | 24h % price change |
| `last_updated` | When the price was last updated |

You can select just these columns to make the table easier to read and work with:

In [11]:
useful_columns = [
    "id",
    "symbol",
    "name",
    "current_price",
    "market_cap",
    "total_volume",
    "price_change_percentage_24h",
    "last_updated",
]

markets_summary = markets_df[useful_columns]
markets_summary

,id,symbol,name,current_price,market_cap,total_volume,price_change_percentage_24h,last_updated
0,bitcoin,btc,Bitcoin,71105.000000,1426495813931,39742423042,-3.32913,2026-06-01T15:48:52.472Z
1,ethereum,eth,Ethereum,1969.050000,237933190535,16231225879,-1.97537,2026-06-01T15:48:52.544Z
2,tether,usdt,Tether,0.998375,187940149618,65167177645,-0.01822,2026-06-01T15:48:52.296Z
3,binancecoin,bnb,BNB,681.020000,91819994568,2873104456,-5.34143,2026-06-01T15:48:52.380Z
4,ripple,xrp,XRP,1.280000,79597193489,1802785294,-3.45527,2026-06-01T15:48:52.228Z


## Pagination — fetching large datasets

CoinGecko tracks thousands of coins — far too many to return in one response. The [`/coins/markets`](https://docs.coingecko.com/v3.0.1/reference/coins-markets) endpoint uses **page-based pagination**:

- `per_page` — how many coins per page (1–250, default 100)
- `page` — which page to fetch (1, 2, 3, …)

In the example below, we fetch the top 20 coins by market cap (4 pages of 5). To fetch everything, keep looping until a page returns fewer records than `per_page` — that means you've reached the end.

In [14]:
import time

all_coins = []
page = 1
per_page = 5
num_pages = 4

while page <= num_pages:
    response = requests.get(
        f"{base_url}coins/markets",
        params={
            "vs_currency": "usd",
            "order": "market_cap_desc",
            "per_page": per_page,
            "page": page,
        },
    )
    response.raise_for_status()
    data = response.json()

    if not data:
        break

    all_coins.extend(data)
    print(f"Page {page}: fetched {len(data)} coins (total so far: {len(all_coins)})")

    if len(data) < per_page:  # last page
        break

    page += 1
    time.sleep(2)  # pause between requests to avoid rate limits

all_coins_df = pd.DataFrame(all_coins)[useful_columns]
all_coins_df

HTTPError: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=5&page=1

### Different types of pagination

The CoinGecko example above is **page-based** pagination — simple and predictable. But exchange APIs rarely agree on one approach — each provider implements pagination differently. The loop structure is always the same (request → append results → update params → repeat), but **what you pass on the next request** varies.

| Type | What you change each loop | Stop when | Examples |
|------|---------------------------|-----------|----------|
| **Page-based** | Increment `page` | Empty page, or `page >= totalPage`, or fewer rows than page size | CoinGecko, KuCoin, Solscan |
| **Offset-based** | Increment `offset` by `limit` | Fewer rows than `limit` | Binance |
| **Cursor-based** | Pass `cursor` / `nextPageCursor` from the response | No cursor returned | Kraken, Bybit, GRVT |
| **Timestamp-based** | Move `startTime` / `after` / `end` using the last record's timestamp | Reached the start of your date range, or fewer rows than limit | OKX, Poloniex, Coinbase, Hyperliquid |
| **ID-based** | Pass last record's `id + 1` as `from` | Fewer rows than limit, or past end time | Huobi |
| **Combined** | Outer loop: split date range into chunks; inner loop: page/offset/cursor | Both loops reach their stop conditions | Bybit, Gate, Bitget |

Always read the provider's API docs — parameter names differ even when the pattern is the same (`page` vs `currentPage`, `limit` vs `pageSize` vs `size`).

The cells below show simplified examples of each pattern. These are illustrative — they won't run against live exchange APIs without your API keys.

#### 1. Page-based — KuCoin

KuCoin returns page metadata in every response: `currentPage`, `totalPage`, and `pageSize`. You increment the page until you've reached the last one.

Docs: [KuCoin deposit history](https://www.kucoin.com/docs-new/rest/account-info/deposit/get-deposit-history)

In [ ]:
# Example: KuCoin page-based pagination
start_ms, end_ms = 0, 0  # placeholder — set to your date range in milliseconds
all_records = []
current_page = 1
page_size = 500

while True:
    params = {
        "startAt": start_ms,
        "endAt": end_ms,
        "pageSize": page_size,
        "currentPage": current_page,
    }
    # response = requests.get(url, params=params, headers=headers)
    # records, page_no, total_pages = parse_response(response)
    # all_records.extend(records)
    # if page_no >= total_pages or len(records) == 0:
    #     break
    # current_page += 1
    print("KuCoin pattern: increment currentPage until page_no >= totalPage")
    break  # remove this line and uncomment above when using a live API key

#### 2. Offset-based — Binance

Binance uses `offset` + `limit` within a fixed time window (`startTime` / `endTime`). Each request skips the first N records; you increase offset by limit until a page comes back short.

Docs: [Binance deposit history](https://developers.binance.com/docs/wallet/deposit-history)

In [ ]:
# Example: Binance offset-based pagination
start_ms, end_ms = 0, 0  # placeholder date range
all_records = []
offset = 0
limit = 500

while True:
    params = {
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": limit,
        "offset": offset,
    }
    # response = requests.get(url, params=params, headers=headers)
    # records = parse_response(response)
    # all_records.extend(records)
    # if len(records) < limit:
    #     break
    # offset += limit
    print("Binance pattern: increment offset by limit until a short page is returned")
    break

#### 3. Cursor-based — Kraken & Bybit

Instead of a page number or offset, the API returns an opaque **cursor** string (e.g. `next_cursor`, `nextPageCursor`). You pass it back on the next request. When the cursor is empty or missing, you're done.

This is common when the server doesn't want you calculating positions yourself — it tracks where you left off.

Docs: [Kraken deposit status](https://docs.kraken.com/api/docs/rest-api/get-deposit-status) · [Bybit deposit records](https://bybit-exchange.github.io/docs/v5/asset/deposit/deposit-record)

In [ ]:
# Example: Kraken cursor-based pagination
all_records = []
params = {"limit": 500, "cursor": True}  # first request asks for a cursor

while True:
    # response = requests.post(url, data=params, headers=headers)
    # records, next_cursor = parse_response(response)
    # all_records.extend(records)
    # if not records or not next_cursor:
    #     break
    # params = {"cursor": next_cursor}
    print("Kraken pattern: pass next_cursor from each response until it is empty")
    break

#### 4. Timestamp-based — OKX & Poloniex

Some APIs don't give you page numbers or cursors at all. Instead, you move a **timestamp boundary** using the last record fetched:

- **OKX** — fetches newest-first. After each page, set `after` to the last record's timestamp + 1, until `after` reaches `before` (your start time).
- **Poloniex** — fetches oldest-first within a window. After each page, set `end` to the last record's timestamp − 1, until you reach `start`.
- **Hyperliquid** — if a full page (500 rows) is returned, advance `startTime` to the max timestamp in the page + 1.

The key detail: timestamps are often **inclusive**, so you add or subtract 1 to avoid fetching the same record twice.

Docs: [OKX deposit history](https://www.okx.com/docs-v5/en/#funding-account-rest-api-get-deposit-history)

In [ ]:
# Example: OKX timestamp-based pagination (newest → oldest)
start_ms, end_ms = 0, 0  # placeholder date range
all_records = []
params = {
    "limit": "100",
    "before": str(start_ms),
    "after": str(end_ms),
}

while int(params["after"]) > int(params["before"]):
    # response = requests.get(url, params=params, headers=headers)
    # records = parse_response(response)
    # all_records.extend(records)
    # if len(records) < int(params["limit"]):
    #     break
    # params["after"] = str(int(records[-1]["ts"]) + 1)
    print("OKX pattern: move 'after' to last record timestamp + 1 until before start")
    break

#### 5. ID-based — Huobi

Huobi paginates by record **ID** rather than timestamp or page number. After each batch, set `from` to the last record's `id + 1` (the API treats `from` as inclusive, so +1 avoids duplicates).

Docs: [Huobi deposit/withdrawal query](https://huobiapi.github.io/docs/spot/v1/en/#search-for-existed-withdraws-and-deposits)

In [ ]:
# Example: Huobi ID-based pagination
all_records = []
params = {"size": 50}

while True:
    # response = requests.get(url, params=params, headers=headers)
    # records = parse_response(response)
    # all_records.extend(records)
    # if len(records) < params["size"]:
    #     break
    # params["from"] = records[-1]["id"] + 1
    print("Huobi pattern: set 'from' to last record id + 1 until a short page is returned")
    break

#### 6. Combined — Bybit, Gate & Bitget

Some exchanges impose **both** a maximum date range per request (often 30 days) **and** pagination within that range. This means **two nested loops**:

1. **Outer loop** — split your full date range into chunks (e.g. 30-day windows, working backwards)
2. **Inner loop** — paginate within each chunk using a cursor (Bybit, Bitget) or offset (Gate, Binance)

Bybit, for example, steps `endTime` back in 30-day windows, and within each window follows `nextPageCursor` until exhausted.

This is the most complex pattern you'll encounter, but the logic is just the inner and outer loops combined.

In [ ]:
# Example: Bybit combined pagination (time chunks + cursor)
start_ms, end_ms = 0, 0  # placeholder date range
all_records = []
params = {"startTime": start_ms, "endTime": end_ms, "limit": 50}
THIRTY_DAYS_MS = 30 * 24 * 60 * 60 * 1000

while params["endTime"] > start_ms:
    params["startTime"] = max(params["endTime"] - THIRTY_DAYS_MS, start_ms)

    while True:
        # response = requests.get(url, params=params, headers=headers)
        # records, next_cursor = parse_response(response)
        # all_records.extend(records)
        # if not next_cursor:
        #     break
        # params["cursor"] = next_cursor
        break  # inner placeholder

    params["endTime"] = params["startTime"]
    if "cursor" in params:
        del params["cursor"]
    print("Bybit pattern: 30-day windows (outer loop) + nextPageCursor (inner loop)")
    break  # outer placeholder

### Practical tips

- **Check the stop condition carefully** — most patterns stop when a page returns fewer rows than the limit, but cursor-based APIs stop when the cursor is absent, and timestamp-based APIs stop when you cross your date boundary.
- **Watch for duplicates** — inclusive timestamps and IDs mean you often need `+ 1` or `− 1` when advancing.
- **Respect rate limits** — add a `time.sleep()` between requests, especially when paginating across many pages.
- **Read the docs first** — the same concept goes by different names (`page` / `currentPage`, `limit` / `pageSize` / `size` / `per_page`).

## API keys and authentication

Public endpoints like CoinGecko's basic price API don't require authentication. But **exchange APIs** and most **blockchain explorer APIs** do — because they're tied to your account and can access private data (your deposit history, your API usage quota, etc.).

You'll typically need an **API key** and **API secret** (and sometimes a **passphrase** you define yourself). Treat these like passwords — never commit them to git or share them in notebooks. Store them in environment variables or a local `.env` file instead, and use **read-only** permissions where possible.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Read the key from an environment variable (set this in your terminal or .env file)
api_key = os.getenv("COINGECKO_API_KEY")

base_url = "https://pro-api.coingecko.com/api/v3/"
method, endpoint = "GET", "coins/bitcoin/history"
request_url = f"{base_url}{endpoint}"

params = {
    "date": "2025-10-06"
}

headers = {
    "x-cg-pro-api-key": api_key
}

response = requests.request(method, request_url, params=params, headers=headers)
response.json()

**Authentication works differently on every venue.** Some exchanges want credentials in a request header, others require you to cryptographically **sign** each request (e.g. with HMAC), and the exact parameter names vary widely. CoinGecko Pro is relatively simple — just an API key in a header — but most exchanges are more involved. There is no single pattern that applies everywhere.

A dedicated section in the course docs will walk through how to authenticate with each exchange step by step — you don't need to memorise the details now.

**Tip: this is a good place to use AI.** When you're ready to connect to a specific exchange, open its API documentation, find the authentication page (and any code example they provide), paste that into an AI assistant, and ask it to implement the signing logic for you. You still need to understand *what* is happening at a high level, but you don't need to write the cryptography yourself.

### One `generate_signature` function

Put all signing logic in one function. It should return a **URL and headers** ready to pass straight to `requests` — e.g. `url, headers = generate_signature(...)` then `requests.get(url, headers=headers)`. Again, this can vary differently from venue to venue, but this is common.

When using AI, paste the venue's auth docs (and any code sample) and ask it to implement *only* this function first.

**Example — Bybit** — [authentication guide](https://bybit-exchange.github.io/docs/v5/guide#authentication) and [official Python sample](https://github.com/bybit-exchange/api-usage-examples/blob/master/V5_demo/api_demo/Encryption_HMAC.py):


In [ ]:
import hashlib
import hmac
import json
import os
import time
import urllib.parse
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("BYBIT_API_KEY")
API_SECRET = os.getenv("BYBIT_API_SECRET")
BASE_URL = "https://api.bybit.com"
RECV_WINDOW = "5000"


def generate_signature(method: str, endpoint: str, params: dict) -> tuple[str, dict]:
    """
    Return (url, headers) for an authenticated Bybit request.
    Based on: https://github.com/bybit-exchange/api-usage-examples/blob/master/V5_demo/api_demo/Encryption_HMAC.py
    """
    timestamp = str(int(time.time() * 1000))

    if method.upper() == "GET":
        query_string = urllib.parse.urlencode(params)
        param_str = timestamp + API_KEY + RECV_WINDOW + query_string
        url = f"{BASE_URL}{endpoint}"
        if query_string:
            url += f"?{query_string}"
    else:
        body = json.dumps(params, separators=(",", ":"))
        param_str = timestamp + API_KEY + RECV_WINDOW + body
        url = f"{BASE_URL}{endpoint}"

    signature = hmac.new(
        API_SECRET.encode("utf-8"),
        param_str.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()

    headers = {
        "X-BAPI-API-KEY": API_KEY,
        "X-BAPI-SIGN": signature,
        "X-BAPI-SIGN-TYPE": "2",
        "X-BAPI-TIMESTAMP": timestamp,
        "X-BAPI-RECV-WINDOW": RECV_WINDOW,
    }
    if method.upper() == "POST":
        headers["Content-Type"] = "application/json"

    return url, headers


def fetch_deposits():
    method = "GET"
    endpoint = "/v5/asset/deposit/query-record"
    params = {"limit": 5}

    url, headers = generate_signature(method, endpoint, params)
    response = requests.request(method, url, headers=headers)
    print(f"Status: {response.status_code}")
    return response.json()


## Rate limits

APIs limit how many requests you can make per minute or per day. This prevents abuse and keeps servers stable.

If you exceed the limit, you'll get a **429** status code. The fix is to **slow down** — add a short pause between requests:

```python
import time

for page in range(1, 10):
    response = requests.get(url, params={"page": page})
    # ... process data ...
    time.sleep(0.5)  # wait half a second between requests
```

Always check the provider's documentation for their specific limits. CoinGecko's free tier, for example, allows roughly 10–30 calls per minute depending on the endpoint.
Generally, there should be a 'Rate Limits' tab on the API documentation (or search 'Rate Limits' in the documentation search bar, if present).

## Handling errors gracefully

In production scripts (and even in notebooks), it's good practice to check whether a request succeeded before trying to parse the data:

In [ ]:
response = requests.get(
    "https://api.coingecko.com/api/v3/simple/price",
    params={"ids": "bitcoin", "vs_currencies": "usd"}
)

if response.status_code == 200:
    price_data = response.json()
    print(f"BTC: ${price_data['bitcoin']['usd']:,.2f}")
elif response.status_code == 429:
    print("Rate limited — wait a moment and try again.")
else:
    print(f"Request failed with status {response.status_code}")
    print(response.text)

## Summary

- An **API** lets you fetch data programmatically instead of using a website's export buttons.
- You send a **request** to a **URL** and receive a **JSON** response.
- Python's `requests` library makes this straightforward: `requests.get(url, params=...)`.
- Use `response.json()` to parse the response, then `pd.DataFrame(...)` to get a table.
- **Large datasets** require **pagination** — exchanges use page-based, offset, cursor, timestamp, ID, or combined approaches; always check the docs for the stop condition.
- **API keys** authenticate you; keep them secret and read-only where possible.
- Respect **rate limits** — add delays between requests if needed.

**Next steps:** Try fetching live data from CoinGecko, then explore the API documentation for an exchange you use regularly. The patterns in this notebook — request, parse, paginate, DataFrame — apply everywhere.